### Set Up

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import sys
print(sys.executable)

C:\Users\PRASHANTH N\PycharmProjects\MTechSem2\gpu\.venv\Scripts\python.exe


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random
from pathlib import Path
import string
import keras_hub

In [ ]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

### Load Dataset

In [9]:
# Train - Test - Val Split Placeholder Location
train_dir = pathlib.Path("../chapter14/imdb_train")
test_dir = pathlib.Path("../chapter14/imdb_test")
val_dir = pathlib.Path("../chapter14/imdb_val")

In [13]:
# Loading the IMDb dataset for use with Keras
batch_size = 8
train_ds = keras.utils.text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(test_dir, batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


### Get Model

In [14]:
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en") # Model Arch & Weights
tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en") # Model Tokenizer
#root = Path(r"C:\\Users\PRASHANTH N\.cache\kagglehub\models\keras\roberta\keras\roberta_base_en\3")
#tokenizer = keras_hub.models.RobertaTokenizer(vocabulary=str(root / "assets" / "tokenizer" / "vocabulary.json"), merges=str(root / "assets" / "tokenizer" / "merges.txt"))

100%|██████████| 445/445 [00:00<00:00, 763kB/s]


100%|██████████| 474M/474M [00:32<00:00, 15.4MB/s]


100%|██████████| 0.99M/0.99M [00:01<00:00, 750kB/s]


100%|██████████| 446k/446k [00:01<00:00, 407kB/s]


In [15]:
tokenizer("The quick brown fox")

Array([  133,  2119,  6219, 23602], dtype=int32)

In [16]:
tokenizer.vocabulary_size(), tokenizer.start_token_id, tokenizer.end_token_id, tokenizer.pad_token_id

(50265, 0, 2, 1)

In [17]:
backbone.summary()

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings          │ (None, None, 768) │ 38,996,736 │ token_ids[0][0]   │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_layer_n… │ (None, None, 768) │      1,536 │ embeddings[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_dropout  │ (None, None, 768) │          0 │ embeddings_layer… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_0 │ (None, None, 768) │  7,087,872 │ embeddings_dropo… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_1 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_2 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_3 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_4 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_5 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_6 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_7 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_8 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_9 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_… │ (None, None, 768) │  7,087,872 │ transformer_laye

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

### Data Processing

In [18]:
# Preprocessing IMDb movie reviews with RoBERTa’s tokenizer

def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label
"""

MAX_LEN = 512

def preprocess(text, label):
    # Convert TensorFlow tensor to Python string
    text = text.numpy().decode("utf-8")
    encoding = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN, return_attention_mask=True, )
    return {
        "token_ids": tf.constant(encoding["input_ids"], dtype=tf.int32),
        "padding_mask": tf.constant(encoding["attention_mask"], dtype=tf.bool),
    }, label

"""
preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

In [19]:
next(iter(preprocessed_train_ds))

({'token_ids': <tf.Tensor: shape=(8, 512), dtype=int32, numpy=
  array([[    0, 12667,  1459, ...,     1,     1,     1],
         [    0, 12582, 14189, ...,     1,     1,     1],
         [    0,   100,  2145, ...,     1,     1,     1],
         ...,
         [    0,  6179,    64, ...,     1,     1,     1],
         [    0,   133,  6644, ...,     1,     1,     1],
         [    0,   100,   269, ...,     1,     1,     1]], dtype=int32)>,
  'padding_mask': <tf.Tensor: shape=(8, 512), dtype=bool, numpy=
  array([[ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         ...,
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False]])>},
 <tf.Tensor: shape=(8,), dtype=int32, numpy=array([0, 1, 0, 0, 1, 1, 0, 0], dtype=int32)>)

### Fine Tune Model

In [20]:
# Adding head
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = keras.layers.Dropout(0.1)(x)
x = keras.layers.Dense(768, activation="relu")(x)
x = keras.layers.Dropout(0.1)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)
classifier.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roberta_backbone    │ (None, None, 768) │ 124,052,7… │ padding_mask[0][… │
│ (RobertaBackbone)   │                   │            │ token_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 768)       │          0 │ roberta_backbone… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 768)       │          0 │ get_item[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 768)       │    590,592 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 768)       │          0 │ dense[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        769 │ dropout_13[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 124,644,097 (475.48 MB)

 Trainable params: 124,644,097 (475.48 MB)

 Non-trainable params: 0 (0.00 B)

### Training

In [21]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"])

classifier.fit(preprocessed_train_ds, epochs=1, validation_data=preprocessed_val_ds)

2500/2500 ━━━━━━━━━━━━━━━━━━━━ 2154s 850ms/step - accuracy: 0.5016 - loss: 0.7036 - val_accuracy: 0.5000 - val_loss: 0.6941


### Inference

In [22]:
classifier.evaluate(preprocessed_test_ds)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 803s 256ms/step - accuracy: 0.5000 - loss: 0.6941


[0.6940646171569824, 0.5]